# 📉 Module 6: Regression Intuition — Real-Life Cases

**The Story:** PizzaChain's CEO asks: "If I increase ad spend by $50, how much more will a store sell?"

Correlation told us ad spend and sales move together. Regression tells us **by how much** — and lets us **predict**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)
n = 50

ad_spend = np.random.normal(200, 50, n)
daily_sales = 150 + 1.8 * ad_spend + np.random.normal(0, 40, n)

stores = pd.DataFrame({'ad_spend': np.round(ad_spend, 1), 'daily_sales': np.round(daily_sales, 1)})
print(f'Dataset: {len(stores)} stores')
stores.head(10)

---
## Case 1: Fitting the Line — y = mx + b

**Formula:** daily_sales = slope × ad_spend + intercept

- **slope (m):** For every $1 more in ad spend, sales change by $m
- **intercept (b):** Predicted sales when ad spend = $0 (theoretical baseline)

In [ ]:
slope, intercept, r_value, p_value, std_err = stats.linregress(stores['ad_spend'], stores['daily_sales'])

print('📊 LINEAR REGRESSION: STEP-BY-STEP')
print('=' * 55)
print(f'  Best-fit line: sales = {slope:.2f} × ad_spend + {intercept:.1f}')
print(f'  Slope     = {slope:.2f} (each $1 in ads → ${slope:.2f} more sales)')
print(f'  Intercept = {intercept:.1f} (baseline sales with $0 ads)')
print(f'  R²        = {r_value**2:.3f} ({r_value**2*100:.1f}% of sales variation explained by ads)')
print(f'  p-value   = {p_value:.6f}')
print()
print(f'💡 INFERENCE:')
print(f'   Every $1 spent on ads generates ~${slope:.2f} in sales.')
print(f'   So $50 more in ads → ~${50*slope:.0f} more in daily sales.')
print(f'   R² = {r_value**2:.2f} means ads explain {r_value**2*100:.0f}% of sales differences.')
print(f'   The other {(1-r_value**2)*100:.0f}% is due to location, staff, weather, etc.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: scatter + regression line
x_line = np.linspace(stores['ad_spend'].min(), stores['ad_spend'].max(), 100)
y_line = slope * x_line + intercept

axes[0].scatter(stores['ad_spend'], stores['daily_sales'], color='#22d3a7', alpha=0.6, s=50, edgecolors='white')
axes[0].plot(x_line, y_line, color='#f45d6d', linewidth=2.5, label=f'y = {slope:.1f}x + {intercept:.0f}')

# Show prediction for $250 ad spend
pred_x = 250
pred_y = slope * pred_x + intercept
axes[0].scatter([pred_x], [pred_y], color='#f5b731', s=150, zorder=5, marker='*', label=f'Prediction: ${pred_x} ads → ${pred_y:.0f} sales')
axes[0].axhline(pred_y, color='#f5b731', linewidth=1, linestyle=':', alpha=0.5)
axes[0].axvline(pred_x, color='#f5b731', linewidth=1, linestyle=':', alpha=0.5)

axes[0].set_xlabel('Ad Spend ($)')
axes[0].set_ylabel('Daily Sales ($)')
axes[0].set_title('Linear Regression: Predicting Sales from Ad Spend', fontsize=12, fontweight='bold')
axes[0].legend(fontsize=9)

# Right: residuals
predicted = slope * stores['ad_spend'] + intercept
residuals = stores['daily_sales'] - predicted
colors_r = ['#22d3a7' if r > 0 else '#f45d6d' for r in residuals]
axes[1].bar(range(len(residuals)), residuals, color=colors_r, alpha=0.7, edgecolor='white')
axes[1].axhline(0, color='white', linewidth=1)
axes[1].set_xlabel('Store #')
axes[1].set_ylabel('Residual (actual − predicted)')
axes[1].set_title('Residuals: Random = Good Model ✅', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'💡 Residuals look random (no pattern) → the linear model is a good fit.')
print(f'   If you saw a curve or funnel, you\'d need a more complex model.')

---
## Case 2: Overfitting — When the Model Memorizes Noise

**The danger:** A complex model can fit training data perfectly but fail on new data.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
x = stores['ad_spend'].values
y = stores['daily_sales'].values
x_smooth = np.linspace(x.min(), x.max(), 200)

titles = ['Underfit (degree 0)', 'Good Fit (degree 1)', 'Overfit (degree 15)']
degrees = [0, 1, 15]
colors = ['#f5b731', '#22d3a7', '#f45d6d']

for i, (deg, title, c) in enumerate(zip(degrees, titles, colors)):
    axes[i].scatter(x, y, color='#7c6aff', alpha=0.5, s=30)
    coeffs = np.polyfit(x, y, deg)
    y_fit = np.polyval(coeffs, x_smooth)
    if deg == 15:
        y_fit = np.clip(y_fit, y.min() - 100, y.max() + 100)
    axes[i].plot(x_smooth, y_fit, color=c, linewidth=2.5)
    
    # Calculate R² on training data
    y_pred = np.polyval(coeffs, x)
    ss_res = np.sum((y - y_pred)**2)
    ss_tot = np.sum((y - y.mean())**2)
    r2 = 1 - ss_res/ss_tot
    
    axes[i].set_title(f'{title}\nR² = {r2:.3f}', fontsize=11, fontweight='bold', color=c)
    axes[i].set_xlabel('Ad Spend ($)')
    if i == 0: axes[i].set_ylabel('Daily Sales ($)')

plt.tight_layout()
plt.show()

print('💡 INFERENCE:')
print('   Underfit: Too simple — misses the obvious trend.')
print('   Good fit: Captures the real pattern without chasing noise.')
print('   Overfit: R² looks great on training data, but those wiggles are noise.')
print('   On NEW data, the overfit model would perform WORSE than the simple line.')

---
## 📝 Summary

| Concept | What It Tells You | Key Number |
|---|---|---|
| Slope | How much Y changes per unit of X | $1.8 more sales per $1 ads |
| Intercept | Baseline Y when X = 0 | ~$150 sales with no ads |
| R² | How much variation is explained | 0.75 = 75% explained |
| Residuals | How wrong each prediction is | Random = good model |
| Overfitting | Model memorizes noise | High train R², low test R² |